# Phase 1: Machine Learning Model Training — Baseline & Tree Models


# Import Libraries


In [1]:
import os
import json
import joblib
import numpy as np
import pandas as pd
from sklearn.dummy import DummyRegressor
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor


# Load Prepared Data


In [2]:
DATA_DIR = "../../data/GeneratedOutput" if os.path.exists("../../data") else "data/GeneratedOutput"
train_path = os.path.join(DATA_DIR, "02.Preprocessing&FeatureSelection", "train.parquet")
val_path = os.path.join(DATA_DIR, "02.Preprocessing&FeatureSelection", "val.parquet")

train_data = pd.read_parquet(train_path)
val_data = pd.read_parquet(val_path)

print(f"Training data shape  : {train_data.shape[0]:,} rows x {train_data.shape[1]} columns")
print(f"Validation data shape: {val_data.shape[0]:,} rows x {val_data.shape[1]} columns")


Training data shape  : 1,973,731 rows x 45 columns
Validation data shape: 422,943 rows x 45 columns


# Select Features and Target


In [3]:
# Select Features and Target
metadata_path = os.path.join(DATA_DIR, "02.Preprocessing&FeatureSelection", "preprocessing_metadata.json")
with open(metadata_path) as f:
    features = json.load(f)["input_features_ordered"]

target = "ccme_values"

X_train = train_data[features]
y_train = train_data[target]

X_val = val_data[features]
y_val = val_data[target]

print(f"Selected {len(features)} input features (8 base + 30 engineered) in canonical metadata order.")
print(f"Target column: {target}")


Selected 38 input features (8 base + 30 engineered) in canonical metadata order.
Target column: ccme_values


# Train a Mean Baseline Regressor


In [4]:
baseline_model = DummyRegressor(strategy="mean")
baseline_model.fit(X_train, y_train)

print(f"Mean baseline fitted. Constant prediction value: {baseline_model.constant_[0][0]:.4f}")


Mean baseline fitted. Constant prediction value: 85.0459


# Train Random Forest Regressor


In [5]:
rf_model = RandomForestRegressor(
    n_estimators=30,
    max_depth=14,
    min_samples_leaf=5,
    random_state=42,
    n_jobs=-1,
)

print(f"Training Random Forest on full {len(X_train):,} rows...")
rf_model.fit(X_train, y_train)
print("Random Forest training complete.")

Training Random Forest on full 1,973,731 rows...
Random Forest training complete.


# Train XGBoost Regressor


In [6]:
xgb_model = XGBRegressor(
    n_estimators=150,
    max_depth=6,
    learning_rate=0.08,
    subsample=0.8,
    colsample_bytree=0.8,
    tree_method="hist",
    early_stopping_rounds=20,
    random_state=42,
    n_jobs=-1,
)

print("Training XGBoost...")
xgb_model.fit(X_train, y_train, eval_set=[(X_val, y_val)], verbose=50)
print(f"XGBoost training complete. Best iteration: {xgb_model.best_iteration}")

Training XGBoost...
[0]	validation_0-rmse:16.26341
[50]	validation_0-rmse:0.62368
[100]	validation_0-rmse:0.45661
[149]	validation_0-rmse:0.43719
XGBoost training complete. Best iteration: 149


# Save Trained Models and Validation Predictions


In [7]:
output_dir = os.path.join(DATA_DIR, "03.ModelTraining")
os.makedirs(output_dir, exist_ok=True)

rf_path = os.path.join(output_dir, "random_forest_model.joblib")
joblib.dump(rf_model, rf_path)

xgb_path = os.path.join(output_dir, "xgboost_model.json")
xgb_model.save_model(xgb_path)

val_predictions = pd.DataFrame({
    "actual": y_val.to_numpy(),
    "pred_baseline": baseline_model.predict(X_val),
    "pred_rf": rf_model.predict(X_val),
    "pred_xgb": xgb_model.predict(X_val),
})
preds_path = os.path.join(output_dir, "tree_val_predictions.parquet")
val_predictions.to_parquet(preds_path, index=False)

# Export training history & metrics
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

history_records = []
for name, model in [("Mean baseline", baseline_model), ("Random Forest", rf_model), ("XGBoost", xgb_model)]:
    train_pred = model.predict(X_train)
    val_pred = model.predict(X_val)
    history_records.append({
        "model": name,
        "train_rows": len(X_train),
        "val_rows": len(X_val),
        "features_count": len(features),
        "train_mae": mean_absolute_error(y_train, train_pred),
        "train_rmse": np.sqrt(mean_squared_error(y_train, train_pred)),
        "train_r2": r2_score(y_train, train_pred),
        "val_mae": mean_absolute_error(y_val, val_pred),
        "val_rmse": np.sqrt(mean_squared_error(y_val, val_pred)),
        "val_r2": r2_score(y_val, val_pred),
    })
history_df = pd.DataFrame(history_records)
history_path = os.path.join(output_dir, "training_history.csv")
history_df.to_csv(history_path, index=False)

print(f"Saved artifacts to {output_dir}:")
print(f"  • {os.path.basename(rf_path)}")
print(f"  • {os.path.basename(xgb_path)}")
print(f"  • {os.path.basename(preds_path)} ({len(val_predictions):,} rows)")
print(f"  • {os.path.basename(history_path)}")
display(history_df.round(4))

Saved artifacts to ../../data/GeneratedOutput/03.ModelTraining:
  • random_forest_model.joblib
  • xgboost_model.json
  • tree_val_predictions.parquet (422,943 rows)
  • training_history.csv


,model,train_rows,val_rows,features_count,train_mae,train_rmse,train_r2,val_mae,val_rmse,val_r2
0,Mean baseline,1973731,422943,38,14.1307,17.6600,0.0000,14.1297,17.6561,-0.0000
1,Random Forest,1973731,422943,38,0.0470,0.1724,0.9999,0.0521,0.2081,0.9999
2,XGBoost,1973731,422943,38,0.1606,0.4331,0.9994,0.1611,0.4372,0.9994


# Basic Training Check


In [8]:
loaded_rf = joblib.load(rf_path)
sample_rf_pred = loaded_rf.predict(X_val.iloc[:5])

loaded_xgb = XGBRegressor()
loaded_xgb.load_model(xgb_path)
sample_xgb_pred = loaded_xgb.predict(X_val.iloc[:5])

print("Sample actual values   :", np.round(y_val.iloc[:5].to_numpy(), 2))
print("Random Forest predicted:", np.round(sample_rf_pred, 2))
print("XGBoost predicted      :", np.round(sample_xgb_pred, 2))
print("\nAll models loaded successfully and generated valid predictions.")


Sample actual values   : [ 55.89  89.74 100.    52.43 100.  ]
Random Forest predicted: [ 55.7   89.63 100.    52.4  100.  ]
XGBoost predicted      : [ 55.11  89.67 100.    52.72 100.  ]

All models loaded successfully and generated valid predictions.
